In [4]:
# -*- coding: utf-8 -*-
"""
Prediction script for the bounded log-space neural network.

This script performs the following tasks:
1. Loads the input scaler, parameter bounds, and trained model weights.
2. Reads new TrPL + PLQY input data.
3. Applies the same input scaling used during training.
4. Predicts kinetic parameters in log10 space.
5. Converts predicted parameters back to the original physical scale.
6. Saves the prediction results to a CSV file.

The model architecture must be identical to that used during training.
"""

# ==========================================================
# 1. Import packages
# ==========================================================
import joblib
import numpy as np
import pandas as pd

import torch
import torch.nn as nn


# ==========================================================
# 2. Global settings
# ==========================================================
NEW_INPUT_FILE = "TrPL+PLQY_test.xlsx"

MODEL_FILE = "trained_bounded_log_model.pth"
X_SCALER_FILE = "x_scaler.pkl"
BOUNDS_INFO_FILE = "bounds_info.pkl"

OUTPUT_FILE = "Prediction_results.csv"

HIDDEN_DIM = 1800
DROPOUT_RATE = 0.048


# ==========================================================
# 3. Device setting
# ==========================================================
def get_device():
    """
    Select CUDA if available, otherwise use CPU.

    Returns
    -------
    device : torch.device
        Computation device.
    """

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    return device


# ==========================================================
# 4. Load scaler and parameter boundary information
# ==========================================================
def load_preprocessing_information(
    x_scaler_file: str,
    bounds_info_file: str
):
    """
    Load the input scaler and parameter boundary information.

    Parameters
    ----------
    x_scaler_file : str
        File path of the saved StandardScaler.
    bounds_info_file : str
        File path of the saved parameter boundary information.

    Returns
    -------
    x_scaler : StandardScaler
        Fitted input scaler from the training stage.
    output_columns : list
        Names of predicted kinetic parameters.
    param_lower : np.ndarray
        Lower bounds of parameters in original physical scale.
    param_upper : np.ndarray
        Upper bounds of parameters in original physical scale.
    lower_log : np.ndarray
        Lower bounds of parameters in log10 scale.
    upper_log : np.ndarray
        Upper bounds of parameters in log10 scale.
    """

    x_scaler = joblib.load(x_scaler_file)
    bounds_info = joblib.load(bounds_info_file)

    output_columns = bounds_info["output_columns"]
    param_lower = bounds_info["param_lower"]
    param_upper = bounds_info["param_upper"]
    lower_log = bounds_info["lower_log"]
    upper_log = bounds_info["upper_log"]

    print("Output columns:", output_columns)
    print("Parameter lower bounds:", param_lower)
    print("Parameter upper bounds:", param_upper)

    return (
        x_scaler,
        output_columns,
        param_lower,
        param_upper,
        lower_log,
        upper_log
    )


# ==========================================================
# 5. Define bounded MLP model
# ==========================================================
class BoundedMLP(nn.Module):
    """
    Bounded MLP model used for kinetic parameter prediction.

    The model predicts six log10-transformed kinetic parameters:
        log10(Rpop), log10(Rdep), log10(NT),
        log10(Reh), log10(Rdet), log10(RAug)

    The raw output is passed through a sigmoid function and then mapped
    into the predefined physical log10 parameter range.

    This architecture must be identical to the model used during training.
    """

    def __init__(
        self,
        input_dim: int,
        lower_log: np.ndarray,
        upper_log: np.ndarray,
        hidden_dim: int = 1800,
        dropout_rate: float = 0.048
    ):
        super(BoundedMLP, self).__init__()

        self.feature_net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(hidden_dim, 6)
        )

        # Register bounds as buffers so that they automatically move
        # with the model between CPU and GPU.
        self.register_buffer(
            "lower_log",
            torch.tensor(lower_log, dtype=torch.float32)
        )

        self.register_buffer(
            "upper_log",
            torch.tensor(upper_log, dtype=torch.float32)
        )

    def forward(self, x):
        raw_output = self.feature_net(x)

        # Sigmoid constrains the normalized output to [0, 1].
        normalized_output = torch.sigmoid(raw_output)

        # Map normalized output to the predefined log10 parameter range.
        bounded_log_output = (
            self.lower_log
            + normalized_output * (self.upper_log - self.lower_log)
        )

        return bounded_log_output


# ==========================================================
# 6. Load new input data
# ==========================================================
def load_new_input_data(
    input_file: str,
    input_format: str = "x_only"
):
    """
    Load new input data for prediction.

    Parameters
    ----------
    input_file : str
        Excel file containing new input data.
    input_format : str
        Format of the input file.

        Available options:
        1. "x_only"
           The file only contains input features, such as TrPL + PLQY data.

        2. "same_as_training"
           The file has the same format as the training file.
           The first six columns are parameters, and the remaining columns
           are input features.

    Returns
    -------
    X_new_raw : np.ndarray
        New input features in original scale.
    """

    new_data = pd.read_excel(input_file)

    if input_format == "x_only":
        X_new_raw = new_data.values.astype(np.float64)

    elif input_format == "same_as_training":
        X_new_raw = new_data.iloc[:, 6:].values.astype(np.float64)

    else:
        raise ValueError(
            "input_format must be either 'x_only' or 'same_as_training'."
        )

    print("New input shape:", X_new_raw.shape)

    return X_new_raw


# ==========================================================
# 7. Scale new input data
# ==========================================================
def scale_new_input_data(
    X_new_raw: np.ndarray,
    x_scaler
):
    """
    Apply the training-stage input scaler to new input data.

    Parameters
    ----------
    X_new_raw : np.ndarray
        New input features in original scale.
    x_scaler : StandardScaler
        Scaler fitted during training.

    Returns
    -------
    X_new_scaled : np.ndarray
        Scaled new input features.
    """

    X_new_scaled = x_scaler.transform(X_new_raw)

    return X_new_scaled


# ==========================================================
# 8. Load trained model
# ==========================================================
def load_trained_model(
    model_file: str,
    input_dim: int,
    lower_log: np.ndarray,
    upper_log: np.ndarray,
    device: torch.device
):
    """
    Initialize the bounded MLP model and load trained weights.

    Parameters
    ----------
    model_file : str
        File path of the saved model weights.
    input_dim : int
        Number of input features.
    lower_log : np.ndarray
        Lower bounds of parameters in log10 scale.
    upper_log : np.ndarray
        Upper bounds of parameters in log10 scale.
    device : torch.device
        Computation device.

    Returns
    -------
    model : nn.Module
        Trained bounded MLP model.
    """

    model = BoundedMLP(
        input_dim=input_dim,
        lower_log=lower_log,
        upper_log=upper_log,
        hidden_dim=HIDDEN_DIM,
        dropout_rate=DROPOUT_RATE
    ).to(device)

    state_dict = torch.load(
        model_file,
        map_location=device
    )

    model.load_state_dict(state_dict)
    model.eval()

    print("Model loaded successfully.")

    return model


# ==========================================================
# 9. Predict kinetic parameters
# ==========================================================
def predict_parameters(
    model: nn.Module,
    X_new_scaled: np.ndarray,
    lower_log: np.ndarray,
    upper_log: np.ndarray,
    device: torch.device
):
    """
    Predict kinetic parameters using the trained bounded MLP model.

    Parameters
    ----------
    model : nn.Module
        Trained bounded MLP model.
    X_new_scaled : np.ndarray
        Scaled new input features.
    lower_log : np.ndarray
        Lower bounds of parameters in log10 scale.
    upper_log : np.ndarray
        Upper bounds of parameters in log10 scale.
    device : torch.device
        Computation device.

    Returns
    -------
    Y_pred_log : np.ndarray
        Predicted parameters in log10 scale.
    Y_pred_original : np.ndarray
        Predicted parameters in original physical scale.
    """

    X_new_tensor = torch.tensor(
        X_new_scaled,
        dtype=torch.float32
    ).to(device)

    with torch.no_grad():
        Y_pred_log_tensor = model(X_new_tensor)

    Y_pred_log = Y_pred_log_tensor.cpu().numpy().astype(np.float64)

    # The model output is already bounded. This clipping is only used as
    # an additional numerical safeguard.
    Y_pred_log = np.clip(Y_pred_log, lower_log, upper_log)

    # Convert log10 values back to the original physical scale.
    Y_pred_original = 10 ** Y_pred_log

    return Y_pred_log, Y_pred_original


# ==========================================================
# 10. Save prediction results
# ==========================================================
def save_prediction_results(
    Y_pred_log: np.ndarray,
    Y_pred_original: np.ndarray,
    output_columns: list,
    output_file: str
):
    """
    Save predicted kinetic parameters to a CSV file.

    Parameters
    ----------
    Y_pred_log : np.ndarray
        Predicted parameters in log10 scale.
    Y_pred_original : np.ndarray
        Predicted parameters in original physical scale.
    output_columns : list
        Names of predicted kinetic parameters.
    output_file : str
        Output CSV filename.

    Returns
    -------
    result_df : pandas.DataFrame
        DataFrame containing prediction results.
    """

    result_dict = {}

    for i, col in enumerate(output_columns):
        result_dict[f"Pred_log10_{col}"] = Y_pred_log[:, i]
        result_dict[f"Pred_{col}"] = Y_pred_original[:, i]

    result_df = pd.DataFrame(result_dict)
    result_df.to_csv(output_file, index=False, encoding="utf-8-sig")

    print(f"\nPrediction results saved to: {output_file}")

    return result_df


# ==========================================================
# 11. Main workflow
# ==========================================================
def main():
    """
    Main workflow for loading the trained model and predicting new data.
    """

    device = get_device()

    (
        x_scaler,
        output_columns,
        param_lower,
        param_upper,
        lower_log,
        upper_log
    ) = load_preprocessing_information(
        x_scaler_file=X_SCALER_FILE,
        bounds_info_file=BOUNDS_INFO_FILE
    )

    # ------------------------------------------------------
    # Input data format
    #
    # Use input_format="x_only" if the new Excel file only contains
    # input features, such as TrPL + PLQY data.
    #
    # Use input_format="same_as_training" if the new Excel file has the
    # same format as the training file, where the first six columns are
    # parameters and the remaining columns are input features.
    # ------------------------------------------------------
    X_new_raw = load_new_input_data(
        input_file=NEW_INPUT_FILE,
        input_format="x_only"
    )

    X_new_scaled = scale_new_input_data(
        X_new_raw=X_new_raw,
        x_scaler=x_scaler
    )

    input_dim = X_new_raw.shape[1]

    model = load_trained_model(
        model_file=MODEL_FILE,
        input_dim=input_dim,
        lower_log=lower_log,
        upper_log=upper_log,
        device=device
    )

    Y_pred_log, Y_pred_original = predict_parameters(
        model=model,
        X_new_scaled=X_new_scaled,
        lower_log=lower_log,
        upper_log=upper_log,
        device=device
    )

    result_df = save_prediction_results(
        Y_pred_log=Y_pred_log,
        Y_pred_original=Y_pred_original,
        output_columns=output_columns,
        output_file=OUTPUT_FILE
    )

    print("\nFirst 5 prediction results:")
    print(result_df.head())


# ==========================================================
# 12. Run script
# ==========================================================
if __name__ == "__main__":
    main()

Using device: cpu
Output columns: ['γpop', 'γdep', 'NT', 'γeh', 'γdet', 'γAug']
Parameter lower bounds: [1.e-10 1.e-11 1.e+14 1.e-12 1.e+04 1.e-29]
Parameter upper bounds: [1.e-08 1.e-09 1.e+16 1.e-10 1.e+06 1.e-27]
New input shape: (8, 401)
Model loaded successfully.

Prediction results saved to: Prediction_results.csv

First 5 prediction results:
   Pred_log10_γpop     Pred_γpop  Pred_log10_γdep     Pred_γdep  \
0        -9.401866  3.964004e-10        -9.877405  1.326157e-10   
1        -9.599757  2.513291e-10        -9.887093  1.296903e-10   
2        -9.438744  3.641300e-10       -10.237495  5.787681e-11   
3        -9.603925  2.489289e-10       -10.458876  3.476357e-11   
4        -8.922724  1.194748e-09        -9.026734  9.402983e-10   

   Pred_log10_NT       Pred_NT  Pred_log10_γeh      Pred_γeh  Pred_log10_γdet  \
0      15.866440  7.352581e+15      -10.727649  1.872196e-11         4.216415   
1      15.781422  6.045353e+15      -10.576154  2.653666e-11         4.258465   
2  

C:\Users\Scarlett\AppData\Local\Temp\ipykernel_22560\3471479818.py:301: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(
